In [3]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")
df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
print(df.shape)
print(df.columns)
df.rename(columns={'review': 'text'}, inplace=True)

(50000, 2)
Index(['review', 'sentiment'], dtype='object')


# NLP Preprocessing

In [5]:
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\VIRAJ\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\VIRAJ\AppData\Roaming\nltk_data...


True

In [6]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    
    # lowercase
    text = text.lower()
    
    # remove urls
    text = re.sub(r'http\S+', '', text)
    
    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # remove numbers
    text = re.sub(r'\d+', '', text)
    
    # tokenization
    words = text.split()
    
    # remove stopwords
    words = [word for word in words if word not in stop_words]
    
    # lemmatization
    words = [lemmatizer.lemmatize(word) for word in words]
    
    return " ".join(words)

In [7]:
df['clean_text'] = df['text'].apply(clean_text)

df[['text', 'clean_text']].head()

,text,clean_text
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode you...
1,A wonderful little production. <br /><br />The...,wonderful little production br br filming tech...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...


# Train-Test Split

In [8]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Bag of Words


In [9]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

# TF-IDF

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [12]:
def evaluate(model, X_train, X_test, y_train, y_test):
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    print("Model:", model.__class__.__name__)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
    print("-"*50)

In [13]:
print("Using Bag of Words:\n")

evaluate(LogisticRegression(), X_train_bow, X_test_bow, y_train, y_test)
evaluate(MultinomialNB(), X_train_bow, X_test_bow, y_train, y_test)
evaluate(DecisionTreeClassifier(), X_train_bow, X_test_bow, y_train, y_test)

Using Bag of Words:



c:\Users\VIRAJ\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Model: LogisticRegression
Accuracy: 0.8724
Precision: 0.8724751420407759
Recall: 0.8724
F1 Score: 0.8723844698525834
--------------------------------------------------
Model: MultinomialNB
Accuracy: 0.8457
Precision: 0.8457331139012982
Recall: 0.8457
F1 Score: 0.8457023407630452
--------------------------------------------------
Model: DecisionTreeClassifier
Accuracy: 0.7171
Precision: 0.7171653777278137
Recall: 0.7171
F1 Score: 0.71710239050904
--------------------------------------------------


In [14]:
print("Using TF-IDF:\n")

evaluate(LogisticRegression(), X_train_tfidf, X_test_tfidf, y_train, y_test)
evaluate(MultinomialNB(), X_train_tfidf, X_test_tfidf, y_train, y_test)
evaluate(DecisionTreeClassifier(), X_train_tfidf, X_test_tfidf, y_train, y_test)

Using TF-IDF:

Model: LogisticRegression
Accuracy: 0.8846
Precision: 0.8848001290985495
Recall: 0.8846
F1 Score: 0.8845725702064238
--------------------------------------------------
Model: MultinomialNB
Accuracy: 0.852
Precision: 0.8520057466944004
Recall: 0.852
F1 Score: 0.8519952042406225
--------------------------------------------------
Model: DecisionTreeClassifier
Accuracy: 0.7169
Precision: 0.7170287739834537
Recall: 0.7169
F1 Score: 0.7168941143250442
--------------------------------------------------


Insights

 TF-IDF performed better than Bag of Words.
 Logistic Regression gave best accuracy.
 Naive Bayes worked well for text data.
 Decision Tree performed lower due to overfitting.
 Preprocessing improved model performance.

# Final Conclusionn
In this project, I built a complete NLP pipeline for sentiment analysis.I applied preprocessing techniques like stopword removal, lemmatization, and cleaning. Then I used BoW and TF-IDF for feature extraction and trained multiple models.Among all models, Logistic Regression with TF-IDF performed the best.This project helped me understand how important preprocessing and feature engineering are in NLP.